<h1>Part 4 - Experiment Tracking</h1>

# Experiment Tracking and Model Management with MLFlow

MLflow Tracking separates the code that produces experiments from the server that stores and exposes them. In this lab, the notebook runs on your machine and sends runs to an MLflow server running in Docker. The server uses SQLite for metadata and a Docker volume for models and other artifacts. This lightweight setup is suitable for local learning; a shared production deployment would typically replace SQLite with PostgreSQL and the local volume with object storage. See the [MLflow self-hosting documentation](https://mlflow.org/docs/latest/self-hosting/).

## Exploring MLFlow

MLflow setup:
* Tracking server: MLflow 3.16.1 in Docker
* Backend store: SQLite in a persistent Docker volume
* Artifact store: local directory in the same persistent Docker volume

The MLflow UI and REST API are exposed at [http://localhost:5000](http://localhost:5000).

### Start MLflow with Docker

Run the following commands in a terminal before executing the Python cells:

```bash
# Create persistent storage once. It keeps runs and models when the container stops.
docker volume create mlflow-data-course

# Start the tracking server in the background.
docker run --name mlflow-server --detach \
  --publish 127.0.0.1:5000:5000 \
  --volume mlflow-data-course:/mlflow \
  ghcr.io/mlflow/mlflow:v3.16.1 \
  mlflow server \
  --host 0.0.0.0 \
  --port 5000 \
  --backend-store-uri sqlite:////mlflow/mlflow.db \
  --artifacts-destination /mlflow/artifacts
```

`--publish 127.0.0.1:5000:5000` exposes the server only on the host loopback interface, while `--volume` makes the SQLite database and artifacts persistent. Check the server with `docker logs mlflow-server` or open the UI at [http://localhost:5000](http://localhost:5000).

Use `docker stop mlflow-server` when you finish. On the next session, restart the existing container with `docker start mlflow-server`. To rebuild it with another MLflow version, remove only the container with `docker rm -f mlflow-server`; the named volume remains intact.

Connect the Python client to the containerized tracking server. All subsequent experiments, runs, metrics and models will be sent to this URI.

In [ ]:
import mlflow

mlflow.set_tracking_uri("http://localhost:5000")
print(f"tracking URI: '{mlflow.get_tracking_uri()}'")

After this initialization, create a client connected to the tracking API and inspect the available experiments.

By refering to mlflow's [documentation](https://mlflow.org/docs/latest/python_api/mlflow.client.html), create a client and display a list of the available experiments using the search_experiments function. This function could prove useful later to programatically explore experiments (rather than in the UI)

In [ ]:
from mlflow import MlflowClient

...

The server creates a default experiment. Its run metadata is stored in SQLite and its artifacts are stored in the persistent Docker volume.

### Creating an experiment and logging a new run

An experiment is a logical entity regrouping the logs of multiple attempts at solving a same problem, called runs. \
We will now work with the classic sklearn dataset iris. Our goal here is to manage to classify the different iris species. To track our models performance, we will log every attempt as a "run" and create a new experiment "iris-experiment-1" to regroup them.

Lookup the mlflow.run and mlflow.start_run functions [here](https://mlflow.org/docs/latest/python_api/mlflow.html?highlight=start_run#mlflow.start_run) to find out how to manage runs.
Explore [this part](https://mlflow.org/docs/latest/python_api/mlflow.html) to learn more about the log_params, log_metrics and log_artifact functions. Find out how to log sklearn models [here](https://mlflow.org/docs/latest/python_api/mlflow.sklearn.html])

Complete the following in order to log the parameters, interesting metrics and the model. Assign the value returned by `mlflow.sklearn.log_model(...)` to a variable named `model_info`; the registry section below uses `model_info.model_uri` to register that exact logged model.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from datetime import datetime

mlflow.set_experiment("iris-experiment-1")
C = 0.1

run_timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")

with mlflow.start_run(run_name=f"iris-logistic-regression-c-{C}-{run_timestamp}") as run:
    run_id = run.info.run_id

    X, y = load_iris(return_X_y=True)

    params = {"C": C, "random_state": 42}
    ...

    model = LogisticRegression(**params).fit(X, y)
    y_pred = model.predict(X)
    ...

    ...
    print(f"default artifacts URI: '{mlflow.get_artifact_uri()}'")

In [ ]:
client = MlflowClient()
experiments = client.search_experiments()
experiments

Try running the training cell with several parameter values, then compare the resulting runs in the already-running MLflow UI at [http://localhost:5000](http://localhost:5000). The server runs independently of the notebook, so no cell needs to remain blocked.

### Interacting with the model registry

If you are satisfied with the last run's model, register it from the URI returned by MLflow. `model_info.model_uri` identifies the exact model logged during the run, while `iris_lr_model` is its stable name in the Model Registry. If that registered name already exists, MLflow preserves it and creates the next version instead of overwriting the previous one.

In [ ]:
# Reuse the exact URI returned when the model was logged

result = mlflow.register_model(model_info.model_uri, "iris_lr_model")

# Use Case

Now we will get back to our taxi rides use case: 

In [ ]:
from typing import List

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

DATA_FOLDER = "../../data"
train_path = f"{DATA_FOLDER}/yellow_tripdata_2021-01.parquet"
test_path = f"{DATA_FOLDER}/yellow_tripdata_2021-02.parquet"
predict_path = f"{DATA_FOLDER}/yellow_tripdata_2021-03.parquet"

## 1 - Load data

In [ ]:
def load_data(path: str) -> pd.DataFrame:
    """Load."""
    return pd.read_parquet(path)


train_df = load_data(train_path)
train_df.head()

## 2 - Prepare the data

Let's prepare the data to make it Machine Learning ready. \
For this, we need to clean it, compute the target (what we want to predict), and compute some features to help the model understand the data better.

### 2-1 Compute the target

We want to predict a taxi trip duration in minutes. Let's compute it as a difference between the drop-off time and the pick-up time for each trip.

In [ ]:
def compute_target(
    df: pd.DataFrame,
    pickup_column: str = "tpep_pickup_datetime",
    dropoff_column: str = "tpep_dropoff_datetime",
) -> pd.DataFrame:
    """..."""
    df["duration"] = df[dropoff_column] - df[pickup_column]
    df["duration"] = df["duration"].dt.total_seconds() / 60
    return df


train_df = compute_target(train_df)

In [ ]:
train_df["duration"].describe()

Let's remove outliers and reduce the scope to trips between 1 minute and 1 hour

In [ ]:
MIN_DURATION = 1
MAX_DURATION = 60


def filter_outliers(df: pd.DataFrame, min_duration: int = 1, max_duration: int = 60) -> pd.DataFrame:
    """..."""
    return df[df["duration"].between(min_duration, max_duration)]


train_df = filter_outliers(train_df)

### 2-2 Prepare features

#### 2-2-1 Categorical features

Most machine learning models don't work with categorical features. Because of this, they must be transformed so that the ML model can consume them.

In [ ]:
CATEGORICAL_COLS = ["PUlocationID", "DOlocationID"]


def encode_categorical_cols(df: pd.DataFrame, categorical_cols: List[str] = None) -> pd.DataFrame:
    """..."""
    if categorical_cols is None:
        categorical_cols = ["PULocationID", "DOLocationID", "passenger_count"]
    df[categorical_cols] = df[categorical_cols].fillna(-1).astype("int")
    df[categorical_cols] = df[categorical_cols].astype("str")
    return df


train_df = encode_categorical_cols(train_df)

In [ ]:
def extract_x_y(
    df: pd.DataFrame,
    categorical_cols: List[str] = None,
    dv: DictVectorizer = None,
    with_target: bool = True,
) -> dict:
    """..."""
    if categorical_cols is None:
        categorical_cols = ["PULocationID", "DOLocationID", "passenger_count"]
    dicts = df[categorical_cols].to_dict(orient="records")

    y = None
    if with_target:
        if dv is None:
            dv = DictVectorizer()
            dv.fit(dicts)
        y = df["duration"].values

    x = dv.transform(dicts)
    return x, y, dv


X_train, y_train, dv = extract_x_y(train_df)

## 3 - Train model

We train a basic linear regression model to have a baseline performance

In [ ]:
def train_model(x_train: csr_matrix, y_train: np.ndarray) -> LogisticRegression:
    """Train a Linear Regression."""
    lr = LinearRegression()
    lr.fit(x_train, y_train)
    return lr


model = train_model(X_train, y_train)

## 4 - Evaluate model

We evaluate the model on train and test data

### 4-1 On train data

In [ ]:
def predict_duration(input_data: csr_matrix, model: LinearRegression) -> np.array:
    """..."""
    return model.predict(input_data)


def evaluate_model(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """..."""
    return root_mean_squared_error(y_true, y_pred)


prediction = predict_duration(X_train, model)
train_me = evaluate_model(y_train, prediction)
train_me

### 4-2 On test data

In [ ]:
test_df = load_data(test_path)

In [ ]:
test_df = compute_target(test_df)
test_df = encode_categorical_cols(test_df)
X_test, y_test, _ = extract_x_y(test_df, dv=dv)

In [ ]:
y_pred_test = predict_duration(X_test, model)
test_me = evaluate_model(y_test, y_pred_test)
test_me

## 5 - Log Model Parameters to MlFlow

Now that all our development functions are built and tested, let's create a training pipeline and log the training parameters, logs and model to MlFlow.

Create a training flow, log all the important parameters, metrics and model. Try to find what could be important and needs to be logged.

In [ ]:
# Set the experiment name
...

# Start a run
run_name = f"linear-regression-{MIN_DURATION}-{MAX_DURATION}min-{run_timestamp}"
with mlflow.start_run(run_name=run_name) as run:
    run_id = run.info.run_id

    # Set tags for the run
    ...

    # Load data
    ...

    # Compute target
    ...

    # Filter outliers
    ...

    # Encode categorical columns
    ...

    # Extract X and y
    ...

    # Train model
    ...

    # Evaluate model
    ...

    # Evaluate model on test set
    ...

    # Log your model
    ...

    # Register your model in mlfow model registry
    ...

If the model is satisfactory, assign the `champion` alias to the selected version. An alias is a stable name that can be moved to a newer version without changing prediction code.

Create an MLflow client and use [`set_registered_model_alias`](https://mlflow.org/docs/latest/api_reference/python_api/mlflow.client.html#mlflow.client.MlflowClient.set_registered_model_alias) to assign the `champion` alias to the appropriate registered model version.

In [ ]:
client = MlflowClient()
...

## 6 - Predict

We can now use our model to predict on fresh unseen data and forecast what is going to be the duration of a tawi trip depending on trip characteristics.

In [ ]:
registered_model_name = ...
# Load prediction data
predict_df = load_data(predict_path)

# Apply feature engineering
predict_df = encode_categorical_cols(predict_df)
X_pred, _, _ = extract_x_y(predict_df, dv=dv, with_target=False)
# Load the model version currently selected as champion
model_uri = f"models:/{registered_model_name}@champion"
model = mlflow.sklearn.load_model(model_uri)

# Make predictions
y_pred = predict_duration(X_pred, model)
y_pred

## 7 - To go further

If you managed to go this far, you can try solving the use case using an other regression model like [XGBoost](https://xgboost.readthedocs.io/en/stable/) for instance.